# B1 — Đánh giá BM25 tự cài

## Mục tiêu

Đánh giá BM25 tự cài trên SciFact và NFCorpus bằng bốn chỉ số:

- nDCG@10
- Recall@100
- MRR
- MAP

Cấu hình:
- k1 = 1.2
- b = 0.75
- Tokenizer: whitespace
- Số tài liệu truy xuất: Top-100

Sử dụng các hàm đã cài trong bm25.py và chi_so.py.

In [3]:
import sys
import inspect
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import bm25
import chi_so

print("ChiMuc:", inspect.signature(bm25.ChiMuc))
print("tim:", inspect.signature(bm25.ChiMuc.tim))

print("\nCác hàm đánh giá:")
for name in ["ndcg_at_k", "recall_at_k", "reciprocal_rank", "average_precision"]:
    if hasattr(chi_so, name):
        fn = getattr(chi_so, name)
        print(name, inspect.signature(fn))
    else:
        print(name, "Không tìm thấy")

ChiMuc: (kho, tach=None)
tim: (self, truy_van, k=10, k1=1.2, b=0.75)

Các hàm đánh giá:
ndcg_at_k (thu_hang, qrels, k)
recall_at_k (thu_hang, qrels, k)
reciprocal_rank (thu_hang, qrels)
average_precision (thu_hang, qrels) -> float


## Nạp dữ liệu SciFact và NFCorpus

Sử dụng tập test của hai bộ dữ liệu BEIR.

Mỗi bộ dữ liệu gồm:
- Corpus: Các tài liệu để xây dựng chỉ mục BM25.
- Queries: Các truy vấn dùng để tìm kiếm.
- Qrels: Nhãn liên quan dùng để đánh giá kết quả.

In [4]:
from beir.datasets.data_loader import GenericDataLoader

DATA_DIR = PROJECT_ROOT / "phanB" / "du_lieu"

scifact_corpus, scifact_queries, scifact_qrels = GenericDataLoader(
    data_folder=str(DATA_DIR / "scifact")
).load(split="test")

nfcorpus_corpus, nfcorpus_queries, nfcorpus_qrels = GenericDataLoader(
    data_folder=str(DATA_DIR / "nfcorpus")
).load(split="test")

print("SciFact:")
print("Documents:", len(scifact_corpus))
print("Queries:", len(scifact_queries))
print("Qrels:", len(scifact_qrels))

print("\nNFCorpus:")
print("Documents:", len(nfcorpus_corpus))
print("Queries:", len(nfcorpus_queries))
print("Qrels:", len(nfcorpus_qrels))

d:\RAG\tep_kem\.venv\Lib\site-packages\beir\datasets\data_loader.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
100%|██████████| 3633/3633 [00:00<00:00, 181485.75it/s]

SciFact:
Documents: 5183
Queries: 300
Qrels: 300

NFCorpus:
Documents: 3633
Queries: 323
Qrels: 323


## Kiểm tra định dạng dữ liệu cho BM25

Kiểm tra cấu trúc của kho tài liệu và một tài liệu mẫu trước khi xây dựng chỉ mục.

Mục đích là đảm bảo dữ liệu BEIR được chuyển đúng sang định dạng mà lớp ChiMuc sử dụng.

In [5]:
import inspect
import bm25
import kho_do_choi

print("ChiMuc.__init__:")
print(inspect.getsource(bm25.ChiMuc.__init__))

print("\nCác thành phần trong kho_do_choi.py:")
print([
    name for name in dir(kho_do_choi)
    if not name.startswith("_")
])

doc_id, doc = next(iter(scifact_corpus.items()))

print("\nTài liệu SciFact mẫu:")
print("ID:", doc_id)
print("Nội dung:", doc)

ChiMuc.__init__:
    def __init__(self, kho, tach=None):
        self.tach = tach or tach_khoang_trang
        self.kho = kho
        self.do_dai = {}
        self.tf = {}
        self.chi_muc = defaultdict(list)
        self.n = len(kho) 
        self.avgdl = 0  

        # CAN VIET: duyet kho mot luot va dien nam thuoc tinh tren.
        # Moi tai lieu chi duoc duyet mot lan.
        for ma_tai_lieu, van_ban in self.kho.items():
            cac_tu = self.tach(van_ban)

            #luu do dai cua cac tai lieu KHO
            self.do_dai[ma_tai_lieu] = len(cac_tu)

            #tinh tan so cua cac tu trong KHO, co bao nhieu tu khac se co bay nhieu cap key-val
            self.tf[ma_tai_lieu] = Counter(cac_tu)

            for tu in set(cac_tu):
                self.chi_muc[tu].append(ma_tai_lieu)

        self.avgdl = sum(self.do_dai.values()) / self.n if self.n > 0 else 0.0


Các thành phần trong kho_do_choi.py:
['KHO', 'QRELS', 'TRUY_VAN']

Tài liệu SciFact mẫu:
ID: 4983
Nội dung: {

## Chuẩn bị kho tài liệu và xây dựng chỉ mục BM25

Chuyển corpus của BEIR thành dạng `{doc_id: văn_bản}`.

Nối tiêu đề và nội dung để giữ lại thông tin của tài liệu.

Xây dựng một chỉ mục BM25 cho mỗi bộ dữ liệu, sử dụng tokenizer mặc định (`tach_khoang_trang`).

In [6]:
def prepare_corpus(corpus):
    return {
        doc_id: doc["title"] + " " + doc["text"]
        for doc_id, doc in corpus.items()
    }


scifact_kho = prepare_corpus(scifact_corpus)
nfcorpus_kho = prepare_corpus(nfcorpus_corpus)

scifact_index = bm25.ChiMuc(scifact_kho)
nfcorpus_index = bm25.ChiMuc(nfcorpus_kho)

print("SciFact:")
print("Số tài liệu:", scifact_index.n)
print("Độ dài trung bình:", round(scifact_index.avgdl, 2))

print("\nNFCorpus:")
print("Số tài liệu:", nfcorpus_index.n)
print("Độ dài trung bình:", round(nfcorpus_index.avgdl, 2))

SciFact:
Số tài liệu: 5183
Độ dài trung bình: 214.63

NFCorpus:
Số tài liệu: 3633
Độ dài trung bình: 233.77


## Truy xuất Top-100 bằng BM25

Thực hiện tìm kiếm cho toàn bộ truy vấn của SciFact và NFCorpus.

Cấu hình:
- k = 100
- k1 = 1.2
- b = 0.75
- Tokenizer: tach_khoang_trang

Lưu kết quả thứ hạng để sử dụng khi tính các chỉ số đánh giá.

In [7]:
def retrieve_top_100(index, queries):
    results = {}

    for query_id, query_text in queries.items():
        ranking = index.tim(
            query_text,
            k=100,
            k1=1.2,
            b=0.75
        )

        results[query_id] = ranking

    return results


scifact_results = retrieve_top_100(scifact_index, scifact_queries)
nfcorpus_results = retrieve_top_100(nfcorpus_index, nfcorpus_queries)

print("SciFact:")
print("Số truy vấn:", len(scifact_results))
print("Kết quả truy vấn đầu tiên:", len(next(iter(scifact_results.values()))))

print("\nNFCorpus:")
print("Số truy vấn:", len(nfcorpus_results))
print("Kết quả truy vấn đầu tiên:", len(next(iter(nfcorpus_results.values()))))

SciFact:
Số truy vấn: 300
Kết quả truy vấn đầu tiên: 100

NFCorpus:
Số truy vấn: 323
Kết quả truy vấn đầu tiên: 100


## Truy xuất Top-100 với tokenizer whitespace

Sử dụng chỉ mục BM25 đã xây dựng để truy xuất 100 tài liệu cho mỗi truy vấn.

Cấu hình:
- Tokenizer: tach_khoang_trang
- k1 = 1.2
- b = 0.75
- k = 100

Lưu kết quả vào hai file JSON riêng, không ghi đè kết quả Lucene analyzer của B2/B3.

In [9]:
import json

def retrieve_top_100(index, queries):
    results = {}

    for query_id, query_text in queries.items():
        results[query_id] = index.tim(
            query_text,
            k=100,
            k1=1.2,
            b=0.75
        )

    return results


scifact_results_ws = retrieve_top_100(
    scifact_index, scifact_queries
)

nfcorpus_results_ws = retrieve_top_100(
    nfcorpus_index, nfcorpus_queries
)

RESULT_DIR = PROJECT_ROOT / "phanB" / "ket_qua"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

for name, results in [
    ("scifact", scifact_results_ws),
    ("nfcorpus", nfcorpus_results_ws)
]:
    path = RESULT_DIR / f"{name}_b1_whitespace_top100.json"

    with open(path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"{name}: {len(results)} truy vấn")
    print(f"Đã lưu: {path.name}")

scifact: 300 truy vấn
Đã lưu: scifact_b1_whitespace_top100.json
nfcorpus: 323 truy vấn
Đã lưu: nfcorpus_b1_whitespace_top100.json


## Đánh giá BM25 với hai phương pháp tách từ

Sử dụng bốn file kết quả Top-100 đã lưu để tính:

- nDCG@10
- Recall@100
- MRR
- MAP

Giữ nguyên k1 = 1.2 và b = 0.75.

Tất cả chỉ số được tính bằng các hàm tự cài trong chi_so.py.

In [12]:

def load_results(filename):
    with open(RESULT_DIR / filename, "r", encoding="utf-8") as f:
        return json.load(f)

def get_ranking(result):
    if isinstance(result, dict):
        return list(result.keys())

    return [doc_id for doc_id, score in result]
    
def evaluate_results(results, qrels):
    metrics = {
        "nDCG@10": 0,
        "Recall@100": 0,
        "MRR": 0,
        "MAP": 0
    }

    for query_id, relevance in qrels.items():
        ranking = get_ranking(results.get(query_id, []))

        metrics["nDCG@10"] += chi_so.ndcg_at_k(
            ranking, relevance, 10
        )
        metrics["Recall@100"] += chi_so.recall_at_k(
            ranking, relevance, 100
        )
        metrics["MRR"] += chi_so.reciprocal_rank(
            ranking, relevance
        )
        metrics["MAP"] += chi_so.average_precision(
            ranking, relevance
        )

    return {
        name: value / len(qrels)
        for name, value in metrics.items()
    }


experiments = [
    ("SciFact", "Whitespace", scifact_qrels,
     "scifact_b1_whitespace_top100.json"),

    ("SciFact", "Lucene", scifact_qrels,
     "scifact_bm25_top100.json"),

    ("NFCorpus", "Whitespace", nfcorpus_qrels,
     "nfcorpus_b1_whitespace_top100.json"),

    ("NFCorpus", "Lucene", nfcorpus_qrels,
     "nfcorpus_bm25_top100.json")
]

print(
    f"{'Dataset':<12} {'Tokenizer':<12} "
    f"{'nDCG@10':>10} {'Recall@100':>12} "
    f"{'MRR':>10} {'MAP':>10}"
)

for dataset, tokenizer, qrels, filename in experiments:
    results = load_results(filename)
    metrics = evaluate_results(results, qrels)

    print(
        f"{dataset:<12} {tokenizer:<12} "
        f"{metrics['nDCG@10']:>10.4f} "
        f"{metrics['Recall@100']:>12.4f} "
        f"{metrics['MRR']:>10.4f} "
        f"{metrics['MAP']:>10.4f}"
    )

Dataset      Tokenizer       nDCG@10   Recall@100        MRR        MAP
SciFact      Whitespace       0.5723       0.7969     0.5465     0.5379
SciFact      Lucene           0.6830       0.9282     0.6511     0.6403
NFCorpus     Whitespace       0.2697       0.2064     0.4828     0.1155
NFCorpus     Lucene           0.3219       0.2464     0.5287     0.1442
